# Phân tích giỏ hàng bằng Eclat

Thí nghiệm dùng [UCI Online Retail](https://doi.org/10.24432/C5BW33) để tìm những mặt hàng thường xuất hiện trong cùng hóa đơn. Phần khai phá itemset và sinh luật sử dụng mã Julia trong repository.

## Môi trường và tham số

Chạy các ô theo thứ tự. Có thể đặt biến môi trường `JULIA_EXE` để dùng Julia từ đường dẫn riêng.

In [ ]:
from pathlib import Path
import os
import platform
import shutil
import subprocess
import tarfile
import tempfile
from urllib.request import urlretrieve

repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts/run_market_basket.jl").is_file()),
    None,
)
if repo is None:
    repo = Path.cwd() / "eclat-mining"
    if not repo.exists():
        subprocess.run([
            "git", "clone", "https://github.com/AIVIETNAM-AIO-Heart208/eclat-mining.git", str(repo)
        ], check=True)
    if not (repo / "scripts/run_market_basket.jl").is_file():
        raise FileNotFoundError("Cần phiên bản repository có scripts/run_market_basket.jl.")

julia = os.environ.get("JULIA_EXE") or shutil.which("julia")
if julia is None:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("Cài Julia 1.9 trở lên hoặc khai báo JULIA_EXE.")
    runtime = Path(tempfile.gettempdir()) / "eclat-julia-runtime"
    runtime.mkdir(exist_ok=True)
    executable = runtime / "julia-1.11.4/bin/julia"
    if not executable.is_file():
        with tempfile.TemporaryDirectory() as download:
            archive = Path(download) / "julia.tar.gz"
            urlretrieve("https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.4-linux-x86_64.tar.gz", archive)
            with tarfile.open(archive) as source:
                source.extractall(runtime, filter="data")
    julia = str(executable)

subprocess.run([julia, "--version"], check=True)
subprocess.run([julia, f"--project={repo}", "-e", "using Pkg; Pkg.instantiate()"], cwd=repo, check=True)
minsup = "1%"
minconf = "50%"

## Từ hóa đơn đến luật kết hợp

Julia dùng XLSX.jl để đọc trực tiếp từng dòng trong workbook gốc sau khi xác minh SHA-256. Gom các dòng hợp lệ theo `InvoiceNo`, giữ mỗi `StockCode` một lần trong giỏ và ánh xạ mã sang ID nguyên theo thứ tự từ điển.

Loại hóa đơn hủy, dòng thiếu mã, số lượng hoặc đơn giá không hợp lệ và các mã phí hay điều chỉnh đã xác định. Kết quả dưới đây gồm số dòng, hóa đơn, mã hàng qua từng bước và ví dụ hóa đơn 536365.

Eclat tìm itemset theo minsup. Chương trình sinh luật có hướng, lọc theo minconf, xếp hạng theo lift và đếm trực tiếp trên các giỏ để kiểm tra lại 10 luật được hiển thị.

In [ ]:
from IPython.display import Markdown, display

with tempfile.TemporaryDirectory() as directory:
    output = Path(directory) / "online_retail_rules.md"
    subprocess.run([
        julia, "--startup-file=no", "--threads=1", f"--project={repo}",
        str(repo / "scripts/run_market_basket.jl"), str(output), minsup, minconf,
    ], cwd=repo, check=True)
    display(Markdown(output.read_text(encoding="utf-8")))

## Diễn giải

Với minsup 1% và minconf 50%, các luật có lift cao nhất tập trung vào thẻ đánh dấu cây gia vị. Hai chiều của một luật có cùng lift nhưng có thể khác confidence vì số giỏ ở mẫu số khác nhau. Có thể dùng kết quả để chọn bộ sản phẩm bán kèm rồi kiểm tra hiệu quả trên dữ liệu mới.

[Báo cáo phân tích giỏ hàng](../docs/report/market_basket.md) trình bày số liệu và diễn giải từng nhóm luật ở ngưỡng mặc định.